# PPO Walkthrough

Goal: understand the PPO clipped objective for positive and negative advantages, then map it to OpenAI Baselines.


## Paper-to-code map

Official source:

- `learning/rl-foundations/official/repos/baselines/baselines/ppo2/ppo2.py`
- `learning/rl-foundations/official/repos/baselines/baselines/ppo2/model.py`
- `learning/rl-foundations/official/repos/baselines/baselines/ppo2/runner.py`

Runnable local source:

- `learning/rl-foundations/src/ppo_minimal.py`
- `learning/rl-foundations/src/gae.py`


In [ ]:
from __future__ import annotations
import importlib.util
import math
import platform
from pathlib import Path
import torch
import torch.nn.functional as F
REPO = Path.cwd()
if not (REPO / "learning").exists():
    REPO = Path.cwd().parent
print("python", platform.python_version())
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
print("repo", REPO)


## Step 1: clip behavior


In [ ]:
def ppo_clip_loss(ratio, advantage, eps=0.2):
    return -torch.min(ratio * advantage, ratio.clamp(1 - eps, 1 + eps) * advantage)
ratio = torch.tensor([0.5, 0.8, 1.0, 1.2, 1.6])
print("A>0", ppo_clip_loss(ratio, torch.ones_like(ratio)))
print("A<0", ppo_clip_loss(ratio, -torch.ones_like(ratio)))


## Step 2: log-probability version


In [ ]:
logp_old = torch.tensor([-1.0, -1.0, -1.0, -1.0])
logp_new = torch.tensor([-1.5, -1.0, -0.8, -0.4])
adv = torch.tensor([1.0, -1.0, 0.5, -0.5])
ratio = (logp_new - logp_old).exp()
print("ratio", ratio)
print("loss", ppo_clip_loss(ratio, adv).mean())


## Step 3: tiny GAE


In [ ]:
def gae(rewards, values, gamma=1.0, lam=0.95):
    adv = torch.zeros_like(rewards)
    next_adv = torch.tensor(0.0)
    next_value = torch.tensor(0.0)
    for t in reversed(range(len(rewards))):
        delta = rewards[t] + gamma * next_value - values[t]
        next_adv = delta + gamma * lam * next_adv
        adv[t] = next_adv
        next_value = values[t]
    return adv
print(gae(torch.tensor([1.0, 1.0, 1.0]), torch.tensor([0.5, 0.3, 0.2])))


## Step 4: official source smoke check


In [ ]:
official_root = REPO / "learning" / "rl-foundations" / "official" / "repos" / "baselines"
files = [official_root / "baselines" / "ppo2" / "ppo2.py", official_root / "baselines" / "ppo2" / "model.py", official_root / "baselines" / "ppo2" / "runner.py"]
for file in files:
    print(file.relative_to(REPO), "exists=", file.exists())
assert all(file.exists() for file in files)


## Closed-book checks

1. For `A > 0`, explain why high ratios are clipped.
2. For `A < 0`, explain why low ratios are clipped.
3. Explain how GAE trades bias and variance as `lambda` changes.
